# Chapter 1 · Lab 2: Run the full Qwen3-8B and Qwen3-32B models

Use the custom decoder from [Lab 1](lab.ipynb) with **all 36 layers of 8B and all
64 layers of 32B**. Both models use their original weights, vocabulary, and tokenizer.
Read the [dimension table and equations](../background.md) alongside this notebook.

**Route:** download both complete pinned checkpoints → inspect metadata and tensor
shapes → audit the custom mapping → load and generate logits on Spark → inspect outputs.
Utilities stay in [notebook_utils.py](notebook_utils.py). The custom operators and mapping
are defined in [Lab 1](lab.ipynb); Transformers supplies the tokenizer and optional oracle.
The custom forward path uses our `Block`, RMSNorm, RoPE, attention, and SwiGLU.

Complete [Spark setup](../../../shared/SETUP.md), select its Python kernel, and save
the notebook before execution. **Run All executes both full models sequentially in
BF16 on `cuda:0`.** Downloads total about 82 GB; weight storage alone is about
16.38 GB (8B) or 65.52 GB (32B) at runtime. Leave room for loading, cache, workspace,
and the OS on Spark's unified memory. Start with the supplied short text fixture.
Models and GPU caches are released between runs; logits are saved on CPU.

## 1. Download the complete pinned weights

The revisions below are immutable Hub commits. Keep them in the notebook and run
manifests; changing them creates a different experiment. Initial execution downloads
all safetensors shards and tokenizer/config files. Set `DOWNLOAD = False` to reuse
already prepared local snapshots offline. The partial `qwen3-8b-source` directory
used for tiny preparation is not a full-model checkpoint.


In [1]:
from pathlib import Path
import sys

CODE_DIR = next(p / 'chapters/01_reconstruct_qwen3/code'
                for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'chapters/01_reconstruct_qwen3/code/notebook_utils.py').is_file())
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from notebook_utils import (
    COURSE_ROOT, VerificationRun, download_model_checkpoint, describe_checkpoint,
    create_text_fixture, inspect_logits, write_json,
)

MODEL_REVISIONS = {
    '8b': 'b968826d9c46dd6066d109eabc6255188de91218',
    '32b': '9216db5781bf21249d130ec9da846c4624c16137',
}
DOWNLOAD = True
snapshots = {}
for model_case, revision in MODEL_REVISIONS.items():
    destination = COURSE_ROOT / 'models' / f'qwen3-{model_case}' / revision
    snapshots[model_case] = (download_model_checkpoint(model_case, revision, destination)
                             if DOWNLOAD else destination)
    print(model_case, revision, snapshots[model_case])


/home/huangruoyu/workspace/inference-engineer-study-hall/course/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Fetching 12 files:   8%|▊         | 1/12 [00:00<00:04,  2.69it/s]

Fetching 12 files:  25%|██▌       | 3/12 [00:00<00:01,  6.01it/s]

Fetching 12 files:  42%|████▏     | 5/12 [00:00<00:00,  7.52it/s]

...

Fetching 24 files:  92%|█████████▏| 22/24 [06:41<00:06,  3.40s/it]

Fetching 24 files:  96%|█████████▌| 23/24 [06:42<00:02,  2.63s/it]

Fetching 24 files: 100%|██████████| 24/24 [06:42<00:00,  2.10s/it]

Fetching 24 files: 100%|██████████| 24/24 [06:42<00:00, 16.78s/it]

32b 9216db5781bf21249d130ec9da846c4624c16137 /home/huangruoyu/workspace/inference-engineer-study-hall/course/models/qwen3-32b/9216db5781bf21249d130ec9da846c4624c16137


## 2. Inspect metadata, layers, and every weight shape

This reads safetensors headers without allocating the model on the GPU. Expect 36/64
decoder layers and 399/707 tensors, respectively. Check `head_dim=128` explicitly:
32B has a query width of 8,192 while its residual stream is 5,120. Both have eight
KV heads and vocabulary 151,936. The printed weight list includes norms, attention
and MLP projections, embeddings, and the vocabulary head.


In [2]:
metadata = {}
for model_case, snapshot in snapshots.items():
    print(f'\nQwen3-{model_case.upper()}')
    metadata[model_case] = describe_checkpoint(snapshot)
    expected_layers, expected_parameters = {
        '8b': (36, 8_190_735_360), '32b': (64, 32_762_123_264),
    }[model_case]
    assert metadata[model_case]['metadata']['num_hidden_layers'] == expected_layers
    assert metadata[model_case]['metadata']['parameters'] == expected_parameters
    assert len(metadata[model_case]['weights']) == 11 * expected_layers + 3



Qwen3-8B
Checkpoint: /home/huangruoyu/workspace/inference-engineer-study-hall/course/models/qwen3-8b/b968826d9c46dd6066d109eabc6255188de91218

Model metadata:
{
  "model_type": "qwen3",
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "num_hidden_layers": 36,
  "hidden_size": 4096,
  "intermediate_size": 12288,
  "num_attention_heads": 32,
  "num_key_value_heads": 8,
  "head_dim": 128,
  "vocab_size": 151936,
  "hidden_act": "silu",
  "rms_norm_eps": 1e-06,
  "tie_word_embeddings": false,
  "rope_theta": 1000000,
  "parameters": 8190735360
}

Layers (embedding, decoder stack, final norm, vocabulary head):
  model.embed_tokens
  model.layers.0
  model.layers.1
  model.layers.2
  model.layers.3
  model.layers.4
  model.layers.5
  model.layers.6
  model.layers.7
  model.layers.8
  model.layers.9
  model.layers.10
  model.layers.11
  model.layers.12
  model.layers.13
  model.layers.14
  model.layers.15
  model.layers.16
  model.layers.17
  model.layers.18
  model.layers.19
  model.layer

## 3. Reuse the notebook implementation and audit its weight mapping

Load the model and mapping definitions from [Lab 1](lab.ipynb), the single source
of the custom implementation. The helper reads its `qwen3-definition` cells and
executes only imports, classes, and functions; it does not run Lab 1's downloads,
demonstrations, comparisons, or timing experiments. Save Lab 1 after editing its operators.

`TinyQwen3` accepts the checkpoint configuration, so a full 8B or 32B configuration
builds all 36 or 64 layers. The name refers to the introductory model, not a fixed
layer count. The explicit mapping factory also comes from Lab 1. The loader requires
exact tensor coverage and shape agreement and uses no Transformers forward path
for the custom backend.


In [3]:
from notebook_utils import load_notebook_implementation

implementation = load_notebook_implementation(CODE_DIR / 'lab.ipynb')
Qwen3 = implementation.TinyQwen3
mapping = implementation.model_weight_name_mapping

runs = {}
for model_case, revision in MODEL_REVISIONS.items():
    snapshot = snapshots[model_case]
    tokens_path = create_text_fixture(
        snapshot, revision,
        COURSE_ROOT / 'results' / f'qwen3-{model_case}-{revision}-tokens.json')
    run = VerificationRun(
        Qwen3, model_case=model_case, revision=revision, snapshot=snapshot,
        tokens_path=tokens_path, device='cuda:0', dtype='bf16',
        name_map_factory=mapping, notebook_path=CODE_DIR / 'lab2.ipynb',
        output_root=COURSE_ROOT / 'results' / f'p1-full-{model_case}')
    run.prepare_checkpoint(download=False)
    prediction = run.audit()
    write_json(run.run_root / 'checkpoint_metadata.json', metadata[model_case])
    runs[model_case] = run


Snapshot: /home/huangruoyu/workspace/inference-engineer-study-hall/course/models/qwen3-8b/b968826d9c46dd6066d109eabc6255188de91218
Artifacts: /home/huangruoyu/workspace/inference-engineer-study-hall/course/results/p1-full-8b/20260912T222537Z-12da81e1
Audited 399 tensors without allocating full model storage.
{
  "parameters": 8190735360,
  "parameter_bytes": 16381470720,
  "final_kv_bytes": 3833856
}
Snapshot: /home/huangruoyu/workspace/inference-engineer-study-hall/course/models/qwen3-32b/9216db5781bf21249d130ec9da846c4624c16137
Artifacts: /home/huangruoyu/workspace/inference-engineer-study-hall/course/results/p1-full-32b/20260912T222537Z-be3fb651


Audited 707 tensors without allocating full model storage.
{
  "parameters": 32762123264,
  "parameter_bytes": 65524246528,
  "final_kv_bytes": 6815744
}


Inspect each printed parameter/KV budget and its `inventory.json` before loading.
Derive the parameter count independently from the background equation; config arithmetic
alone is not a checkpoint audit. Export a mapping CSV with
`checkpoint_name,engine_name,shape,numel,status` for your submission.

Each snapshot's own tokenizer renders a non-thinking prompt and four forced
continuation tokens; source text, rendered prompt, token IDs, settings, and tokenizer
revision are saved together. Before sharing token IDs between sizes, verify their
vocabulary, special-token, and template semantics.

## 4. Load full weights and generate custom logits

The next cells load one model at a time in BF16. The loader copies tensors from the
shards into the custom module's preallocated storage. It performs prompt prefill,
retains each layer's KV cache, and processes the supplied continuation one token at a
time. It saves one complete vocabulary vector after the prompt and each continuation
token: expected shape **`[5, 151936]`**. The last-position vocabulary projection
reduces output work without skipping any decoder layer.

These are teacher-forced histories for reproducible comparisons, not a free-running
sampled completion. The top logits below suggest the next token at each fixed history;
they do not feed back into the subsequent input.


In [4]:
custom_outputs = {}
custom_outputs['8b'] = runs['8b'].generate_logits('custom')
logits_8b = inspect_logits(custom_outputs['8b'], snapshots['8b'])
assert tuple(logits_8b.shape) == (5, 151936)


Saved 5 complete vocabulary vectors to /home/huangruoyu/workspace/inference-engineer-study-hall/course/results/p1-full-8b/20260912T222537Z-12da81e1/custom


Saved logits [positions, vocabulary]: (5, 151936)
Position 0: [{'token_id': 32, 'text': 'A', 'logit': 38.25}, {'token_id': 82707, 'text': 'KV', 'logit': 28.625}, {'token_id': 785, 'text': 'The', 'logit': 28.125}, {'token_id': 334, 'text': '**', 'logit': 26.625}, {'token_id': 2082, 'text': 'An', 'logit': 26.375}]
Position 1: [{'token_id': 3070, 'text': ' **', 'logit': 29.875}, {'token_id': 84648, 'text': ' KV', 'logit': 26.25}, {'token_id': 353, 'text': ' *', 'logit': 22.375}, {'token_id': 5309, 'text': ' Key', 'logit': 21.625}, {'token_id': 1376, 'text': ' key', 'logit': 21.375}]
Position 2: [{'token_id': 6500, 'text': ' cache', 'logit': 34.5}, {'token_id': 320, 'text': ' (', 'logit': 31.5}, {'token_id': 36680, 'text': '-cache', 'logit': 27.625}, {'token_id': 19479, 'text': ' Cache', 'logit': 27.5}, {'token_id': 3070, 'text': ' **', 'logit': 24.875}]
Position 3: [{'token_id': 320, 'text': ' (', 'logit': 32.25}, {'token_id': 24722, 'text': ' speeds', 'logit': 28.875}, {'token_id': 3070,

In [5]:
# The 8B GPU model/cache have been released; only its saved CPU logits remain.
custom_outputs['32b'] = runs['32b'].generate_logits('custom')
logits_32b = inspect_logits(custom_outputs['32b'], snapshots['32b'])
assert tuple(logits_32b.shape) == (5, 151936)


Saved 5 complete vocabulary vectors to /home/huangruoyu/workspace/inference-engineer-study-hall/course/results/p1-full-32b/20260912T222537Z-be3fb651/custom


Saved logits [positions, vocabulary]: (5, 151936)
Position 0: [{'token_id': 32, 'text': 'A', 'logit': 36.5}, {'token_id': 641, 'text': 'In', 'logit': 31.875}, {'token_id': 785, 'text': 'The', 'logit': 30.75}, {'token_id': 16429, 'text': 'Using', 'logit': 30.625}, {'token_id': 334, 'text': '**', 'logit': 30.0}]
Position 1: [{'token_id': 3070, 'text': ' **', 'logit': 30.5}, {'token_id': 84648, 'text': ' KV', 'logit': 26.875}, {'token_id': 5309, 'text': ' Key', 'logit': 26.5}, {'token_id': 1376, 'text': ' key', 'logit': 25.25}, {'token_id': 82707, 'text': 'KV', 'logit': 24.5}]
Position 2: [{'token_id': 320, 'text': ' (', 'logit': 30.0}, {'token_id': 6500, 'text': ' cache', 'logit': 29.875}, {'token_id': 19479, 'text': ' Cache', 'logit': 26.25}, {'token_id': 3070, 'text': ' **', 'logit': 24.25}, {'token_id': 36680, 'text': '-cache', 'logit': 24.0}]
Position 3: [{'token_id': 320, 'text': ' (', 'logit': 31.75}, {'token_id': 24722, 'text': ' speeds', 'logit': 30.125}, {'token_id': 3070, 'text

For each model, keep `custom/logits.safetensors`, `custom/summary.json`, the strict
inventory, memory prediction, and manifest under its printed run directory.
Manifests hash this notebook and the custom implementation; summaries record the
parameter count and allocated/reserved/peak GPU bytes. Logits are stored as FP32 CPU
tensors after BF16 inference. Cold diagnostic timings include synchronization and
must not be presented as warmed serving benchmarks.

## 5. Optional oracle run; required for full-project numerical acceptance

Generating finite logits shows that loading and execution succeeded. To establish
equivalence, run the Transformers oracle on the same saved histories and compare
**all** logits using `compare_checkpoint_logits()` in [notebook_utils.py](notebook_utils.py), through the same utility
used in Lab 1. Record the Transformers version separately from the checkpoint commit.
Choose BF16 thresholds from independent reference/backend calibration, freeze them
before acceptance, and preserve failures. Tiny FP32 thresholds do not automatically
apply to these full BF16 models.

Run this recipe separately after defining your justified numeric tolerances:

```python
for model_case, run in runs.items():
    run.generate_logits('transformers')
    run.compare(rtol=declared_bf16_rtol, atol=declared_bf16_atol)
```

If a comparison fails, locate the first differing operator/layer. Do not infer
correctness from plausible text or adjust thresholds just to accept the custom model.

## 6. Full-project extensions

### Make cache behavior independent of batching

Add request-specific processed lengths and storage to your engine. Check full prefill,
chunks `[3,1,7]`, eleven one-token calls, and a continuation after prefill. Compare a
request alone and in a mixed-length batch; this requires extending the supplied
equal-length reference. Every cached read must be below the request's processed length.

Test an empty initial cache, one-token prompts, lengths differing by one, a chunk
ending exactly at capacity, and attempted context overflow. Save full-vocabulary
comparisons for 8B's full, incremental, unequal-chunk, and mixed-batch paths, plus
selected short 32B cases. Keep projection dimensions driven by the config.

### Measure and explain the full-model cache hypothesis

At $S=128,512,2048$ and $B=1$, generate a fixed number of new tokens with cached decode
and repeated full-prefix computation. Predict which terms disappear and which grow.
Use at least three independent warmed repetitions and profile a separate run.
Record prefill time, per-step decode, parameter storage, peak loading memory, and
live/reserved device memory. Keep tokenization outside kernel timing.

Compute logical bytes with `sum(t.numel()*t.element_size())`; measure allocator and
host/system memory separately. Repeat a short 32B continuation and representative
memory points. Plot predicted versus measured memory and cached versus recomputed
timing, explaining both the weight intercept and KV slope. Measure one bounded
optimization supported by the evidence, such as avoiding cache concatenation or
redundant head copies. Report scope and uncertainty even when the change is slower.

Keep the [completion checklist in Lab 1](lab.ipynb#6.-Completion-checklist-and-submission)
for both notebooks. Submit both model inventories and numerical artifacts, memory
accounting, raw timing data, one measured intervention, and the 1,200–2,000 word
[research memo](../../../shared/REPORT.md). Save a greedy non-thinking reference
continuation for qualitative inspection alongside the forced-history comparisons.

[Background theory and readings](../background.md) · [Lab 1](lab.ipynb) ·
[Next chapter](../../02_performance_model/background.md)
